# ebook2audiobook - Cloud (Google Colab)

Run the whole conversion (and optional translation) on a free cloud GPU.
This notebook receives the settings chosen in the desktop app automatically
(through the Colab URL). You only upload the ebook file here.

## How to use

1. In the desktop app open the **Cloud** tab and click **"Open Google Colab with current settings"**.
2. Runtime -> Change runtime type -> **T4 GPU** (recommended).
3. Run the three cells below top to bottom.
4. Upload your ebook when prompted; the finished audiobook downloads automatically.

If the settings were not picked up automatically, paste the config string
(copied to your clipboard by the app) into the **config_b64** field in the first cell.

**Tip (phones / flaky connections):** put the ebook into the `ebook2audiobook`
folder of your Google Drive. It survives runtime disconnects, is found
automatically after every reconnect, and the finished audiobook is copied
**Auto-found book?** if a file is already in `MyDrive/ebook2audiobook` (or
was left in `/content`), cell (1) does **not** start converting it right
away. It prints the found file and asks: press **Enter** (or `y`) to continue
with it, or type `u` to upload a new file instead. This prevents silently
converting the wrong book on a phone.

**Google Drive access:** you are asked to allow Google Drive access only
*once per runtime session* (the first time a cell uses the Drive). Colab
remembers the permission for the whole session, so re-running the cells or
converting more books does **not** ask again. A new browser or a fresh
runtime may ask once more - that is Google's own security prompt.


In [ ]:
#@title (1) Settings & upload the ebook { display-mode: "form" }
# Reads the settings sent from the desktop app (via the Colab URL) or from the
# manual field below, then picks up the ebook file.
config_b64 = ""  #@param {type:"string"}
# Ask for Google Drive access only when Drive is actually used.
# Colab grants/remembers the authorization for the whole runtime
# session, so you answer the Google prompt at most once per runtime.
use_drive = True  #@param {type:"boolean"}

import base64, glob, json, os
try:
    from google.colab import output as _cin_output, files as _cin_files
    _IN_COLAB = True
except Exception:
    _IN_COLAB = False

if not config_b64 and _IN_COLAB:
    try:
        import urllib.parse as _up
        _qs = _cin_output.eval_js('window.location.search') or ""
        _params = _up.parse_qs(_qs.lstrip('?'))
        config_b64 = (_params.get('c') or [""])[0]
    except Exception as _e:
        print("Could not read settings from the URL:", _e)

def _decode_cfg(b64):
    if not b64:
        return {}
    b64 = b64.strip().strip('"\'')
    # A full URL can be pasted as well - pull the c= parameter out of it
    if '://' in b64 or 'c=' in b64:
        import urllib.parse as _up2
        _q = _up2.urlparse(b64).query or b64.split('?', 1)[-1]
        _c = _up2.parse_qs(_q).get('c')
        if _c:
            b64 = _c[0]
    b64 = b64.strip()
    b64 += "=" * (-len(b64) % 4)
    try:
        return json.loads(base64.urlsafe_b64decode(b64.encode('ascii')).decode('utf-8'))
    except Exception as _e:
        print("Could not decode the config string:", _e)
        print("Paste the code copied by the app (or the whole Colab link) into config_b64.")
        return {}

CFG = _decode_cfg(config_b64)
globals()['CFG'] = CFG
if CFG:
    print("Settings received from the app:")
    print(json.dumps(CFG, ensure_ascii=False, indent=2))
else:
    print("No settings found - defaults will be used. You can paste the config into the field above.")

# Google Drive keeps the ebook (and the finished audiobook) safe across
# runtime resets: put the book into MyDrive/ebook2audiobook once and it is
# picked up automatically after every reconnect.
DRIVE_DIR = '/content/drive/MyDrive/ebook2audiobook'
# --- Google Drive: mount ONCE per runtime session only. ---------------
# drive.mount() shows Google's authorization screen every time it runs, so
# we guard it with a session flag: the prompt appears only on the very first
# mount of a runtime / first use of a new browser. Re-running cell (1) or a
# later re-connect reuses the already-mounted /content/drive and does NOT ask
# again. If Drive is not needed (book already found in /content), we skip the
# mount entirely so no prompt is shown at all.
_DRIVE_MOUNTED = globals().get('_DRIVE_MOUNTED', False)
if (use_drive and _IN_COLAB and not _DRIVE_MOUNTED
        and not os.path.isdir('/content/drive/MyDrive')):
    try:
        from google.colab import drive as _drive
        _drive.mount('/content/drive')
        _DRIVE_MOUNTED = True
        globals()['_DRIVE_MOUNTED'] = True
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print("Google Drive mounted; drop your books into MyDrive/ebook2audiobook")
    except Exception as _e:
        print("Google Drive not mounted:", _e)
elif _DRIVE_MOUNTED or os.path.isdir('/content/drive/MyDrive'):
    print("Google Drive already attached - no new authorization needed.")

# The ebook can arrive three ways:
#  a) MyDrive/ebook2audiobook on Google Drive - survives runtime resets;
#  b) /content via the Files panel (folder icon -> upload icon) - reliable
#     on phones but wiped on every runtime reset;
#  c) the upload widget below - fine on desktop, flaky on phones.
EBOOK_EXTS = ('epub', 'mobi', 'azw3', 'azw', 'pdf', 'txt', 'rtf', 'docx',
              'doc', 'html', 'htm', 'fb2', 'md', 'odt')

def _find_ebook():
    found = []
    for _e in EBOOK_EXTS:
        found += glob.glob('/content/*.' + _e)
    if not found and os.path.isdir(DRIVE_DIR):
        for _e in EBOOK_EXTS:
            found += glob.glob(DRIVE_DIR + '/*.' + _e)
    return max(found, key=os.path.getmtime) if found else ''

EBOOK = _find_ebook()
# If a book was found automatically (e.g. on Google Drive, or left over
# in /content from a previous upload), DO NOT start converting it right
# away. Ask the user whether to continue with that file or upload a fresh
# one instead - this avoids silently converting the wrong book, which is
# easy to do on a phone.
if EBOOK and _IN_COLAB:
    print("\nНайден файл книги:")
    print("  ", EBOOK)
    _ans = input(
        "Продолжить конвертацию с этим файлом? [Enter/y - да, u - загрузить новый]: "
    ).strip().lower()
    if _ans in ('u', 'upload', 'new', 'n', 'no', 'н', 'новый'):
        EBOOK = ''
if not EBOOK and _IN_COLAB:
    print("\nUpload your ebook file (epub, pdf, mobi, txt, ...).")
    print("PHONE TIP: if the button below is greyed out or says no file was")
    print("chosen, upload the book via the Files panel instead (folder icon")
    print("on the left -> upload icon), then re-run this cell.")
    try:
        _uploaded = _cin_files.upload()
        if _uploaded:
            EBOOK = list(_uploaded.keys())[0]
    except Exception as _e:
        print("Upload widget failed:", _e)
    EBOOK = EBOOK or _find_ebook()

globals()['EBOOK'] = EBOOK
if EBOOK:
    print("\nSelected ebook:", EBOOK)
else:
    print("\nNo ebook found yet - put it into MyDrive/ebook2audiobook on Google")
    print("Drive (best on phones: survives disconnects) or upload it via the")
    print("Files panel, then re-run this cell.")


In [ ]:
#@title (2) Install dependencies & clone the repository
import os, subprocess

CFG = globals().get('CFG', {})
REPO = CFG.get('repo', 'Tarkas/Book-to-audiobook')
BRANCH = CFG.get('branch', 'master')
print(f"Repo: {REPO}  Branch: {BRANCH}")

# Run a shell command streaming its output into the cell line by line.
# Plain subprocess.run() output is often swallowed by Colab, which makes
# failures invisible - this way we actually see what is happening.
def _run_stream(cmd):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         errors='replace')
    for _line in p.stdout:
        print(_line, end='', flush=True)
    p.wait()
    return p.returncode

os_cmds = [
    "sudo apt-get update -qq",
    "apt-get install -y libxcb-cursor0 libegl1 libopengl0",
    "sudo -v && wget -nv -O- https://download.calibre-ebook.com/linux-installer.sh | sudo sh /dev/stdin",
    "apt-get install -y ffmpeg espeak-ng mecab libmecab-dev mecab-ipadic-utf8 nodejs",
    "pip install --no-cache-dir -q mecab-python3 unidic-lite unidic",
    "python -m unidic download",
]
for c in os_cmds:
    print()
    _run_stream(c)

SETUP_REV = "2026-08-26e"  # bump when this cell changes - visible in the log
print(f"Cell setup revision: {SETUP_REV}")

if not os.path.isdir('ebook2audiobook/.git'):
    # Remove stale directory without .git so clone can proceed
    if os.path.isdir('ebook2audiobook'):
        import shutil as _sh
        _sh.rmtree('ebook2audiobook', ignore_errors=True)
        print('Removed stale ebook2audiobook directory (no .git).')
    for _attempt in range(3):
        r = subprocess.run(f"git clone --depth 1 --branch {BRANCH} --no-tags https://github.com/{REPO}.git ebook2audiobook", shell=True,
                           capture_output=True, text=True)
        if r.returncode == 0:
            break
        print(f"Clone attempt {_attempt+1}/3 failed: {r.stderr.strip()}")
        import time; time.sleep(5)
    if r.returncode != 0:
        raise SystemExit(f"git clone failed after 3 attempts (rc={r.returncode}) - check REPO/BRANCH above.")
else:
    # Re-running these cells in the same runtime must NOT reuse a stale
    # clone: sync it to the tip of BRANCH so recent fixes are picked up.
    subprocess.run(f"git -C ebook2audiobook fetch --depth 1 --no-tags --prune origin {BRANCH}", shell=True)
    _ru = subprocess.run("git -C ebook2audiobook reset --hard FETCH_HEAD", shell=True)
    if _ru.returncode != 0:
        print("WARNING: could not update the existing clone - continuing as is.")
_rh = subprocess.run("git -C ebook2audiobook rev-parse --short HEAD", shell=True,
                     capture_output=True, text=True)
print("Repo checkout:", (_rh.stdout or '?').strip())
# Colab has no wheels for the pinned torch==2.1.0 stack (and ships its own
# CUDA torch anyway), so drop those pins - otherwise pip aborts the WHOLE
# install and nothing gets installed (ModuleNotFoundError: ebooklib etc).
_skip = ('torch==', 'torchaudio==', 'torchvision==')
with open('ebook2audiobook/requirements.txt', encoding='utf-8') as _f:
    _reqs = [l for l in _f if not l.strip().startswith(_skip)]

# Colab switched its runtime to Python 3.13, which the repo pins do not
# support: coqui-tts 0.26 and numpy<2 ship no wheels for 3.13 and are
# rejected by pip ('Requires-Python <3.13'). Swap those pins ONLY when
# running on 3.13+ - older runtimes keep using the versions tested on
# Windows/Linux desktop. The combo mirrors upstream ebook2audiobook:
# coqui-tts 0.27.5 (first series with py3.13 support) wants transformers >=4.57.
import sys as _sys
_PY313_PIN_SWAPS = (
    ('coqui-tts',      'coqui-tts[languages]==0.27.5\n'),
    ('transformers==', 'transformers>=4.57,<5\n'),
    ('numpy==',        'numpy>=2.1\n'),
    # kokoro/misaki declare Requires-Python <3.13 although they work fine
    # on 3.13; they are force-installed below, so drop the unresolvable pin.
    ('kokoro>=',       ''),
)
if _sys.version_info[:2] >= (3, 13):
    def _swap_pins(line):
        s = line.strip()
        for prefix, replacement in _PY313_PIN_SWAPS:
            if s.startswith(prefix):
                return replacement
        return line
    _reqs = [_swap_pins(l) for l in _reqs]
    print(f"Python {_sys.version.split()[0]} detected "
          "- swapped pins for py3.13-compatible versions.")

with open('/content/req_colab.txt', 'w', encoding='utf-8') as _f:
    _f.writelines(_reqs)

# kokoro & misaki are pure-Python but their authors capped Requires-Python
# at <3.13 although they run fine on 3.13. Installing them by DIRECT wheel
# URL plus --ignore-requires-python forces pip to bypass the Requires-Python
# metadata inside the wheel for these two packages only. This flag only
# applies to this isolated pip invocation; the main requirements install
# below runs without it, so unrelated deps still resolve normally.
# The pin was dropped from req_colab.txt, so the big install below never
# trips over kokoro. Loud + verified on purpose: a silent failure here used
# to surface much later as a confusing pip error.
if _sys.version_info[:2] >= (3, 13):
    # pip ignores --no-deps when using @URL syntax and resolves deps anyway,
    # letting it try spacy 4.0.0.dev3 (sdist, no py3.13 wheel -> failure).
    # Download the wheels with wget, then install locally with --no-deps.
    _whl_dir = '/tmp/_kokoro_misaki_whls'
    _mkdir_cmd = 'mkdir -p ' + _whl_dir
    _run_stream(_mkdir_cmd)
    _dl_misaki = ('wget -q -O ' + _whl_dir + '/misaki-0.9.4-py3-none-any.whl '
        'https://files.pythonhosted.org/packages/82/ec/0ee4110ddb54278b8f21c40a140370ae8f687036c4edf578316602697c56/misaki-0.9.4-py3-none-any.whl')
    _dl_kokoro = ('wget -q -O ' + _whl_dir + '/kokoro-0.9.4-py3-none-any.whl '
        'https://files.pythonhosted.org/packages/ea/cc/75f41633c75224ba820a4533163bc8b070b6bf25416014074c63284c2d4e/kokoro-0.9.4-py3-none-any.whl')
    print('$ wget: downloading misaki-0.9.4 and kokoro-0.9.4 wheels')
    _run_stream(_dl_misaki)
    _run_stream(_dl_kokoro)
    _install_cmd = (_sys.executable + ' -m pip install --ignore-requires-python --no-deps '
                    + _whl_dir + '/misaki-0.9.4-py3-none-any.whl '
                    + _whl_dir + '/kokoro-0.9.4-py3-none-any.whl')
    print('$ pip install --ignore-requires-python --no-deps (local wheels): misaki[en]==0.9.4 kokoro==0.9.4')
    _rc = _run_stream(_install_cmd)
    if _rc != 0:
        raise SystemExit('kokoro/misaki install FAILED - see output above.')
    import importlib.util as _ilu
    if _ilu.find_spec('kokoro') is None or _ilu.find_spec('misaki') is None:
        raise SystemExit('kokoro/misaki still missing after install.')
    print('kokoro==0.9.4 + misaki==0.9.4 installed (Requires-Python cap bypassed, deps deferred).')

print("\nInstalling python requirements (a few minutes)...")
r = _run_stream("pip install --no-cache-dir -r /content/req_colab.txt")
if r != 0:
    raise SystemExit("pip install of requirements FAILED - see output above.")
print("Requirements installed.")

# pyannote.audio 3.1.1 / speechbrain 1.0.0 (pinned for Windows) may call
# torchaudio.set_audio_backend at import time, but torchaudio >= 2.2
# (Colab's preinstalled build) removed it. Neuter those calls in the
# installed packages - they were no-ops anyway.
import glob as _g, re as _re
_patch_files = []
for _pkg_dir in ('pyannote/audio', 'speechbrain'):
    _patch_files += _g.glob('/usr/local/lib/python3*/dist-packages/' + _pkg_dir + '/**/*.py',
                            recursive=True)
for _p in _patch_files:
    with open(_p, encoding='utf-8') as _f:
        _s = _f.read()
    _s2 = _re.sub(r'^(\s*)torchaudio\.set_audio_backend\(',
                  r'\1pass  # torchaudio>=2.2 removed it (',
                  _s, flags=_re.M)
    if _s2 != _s:
        with open(_p, 'w', encoding='utf-8') as _f:
            _f.write(_s2)
        print("Patched set_audio_backend in", _p)

print("\nInstalled versions:")
from importlib.metadata import version as _pkgver
for _pkn in ('kokoro', 'misaki', 'coqui-tts', 'transformers', 'numpy',
             'torch', 'torchaudio', 'gradio'):
    try:
        print(f'  {_pkn}=={_pkgver(_pkn)}')
    except Exception:
        print(f'  {_pkn}: NOT INSTALLED')
print("\nSetup done.")

In [ ]:
#@title (3) Convert (and translate) in the cloud + download
import os, glob, subprocess

CFG = globals().get('CFG', {})
EBOOK = globals().get('EBOOK', '')
if not EBOOK or not os.path.exists(EBOOK):
    # Fall back to whatever was dropped into /content via the Files panel
    _exts = ('epub', 'mobi', 'azw3', 'azw', 'pdf', 'txt', 'rtf', 'docx',
             'doc', 'html', 'htm', 'fb2', 'md', 'odt')
    _found = []
    for _e in _exts:
        _found += glob.glob('/content/*.' + _e)
    EBOOK = max(_found, key=os.path.getmtime) if _found else ''
# Also search in the Drive checkpoint directory (proc folder may contain the ebook)
import hashlib as _h
_ckpt_search = ''
if not EBOOK or not os.path.exists(EBOOK):
    if os.path.isdir('/content/drive/MyDrive'):
        _drive_ckpt_base = '/content/drive/MyDrive/ebook2audiobook/checkpoints'
        if os.path.isdir(_drive_ckpt_base):
            for _ckpt_id in os.listdir(_drive_ckpt_base):
                _ckpt_proc = os.path.join(_drive_ckpt_base, _ckpt_id, 'proc')
                if os.path.isdir(_ckpt_proc):
                    for _e in _exts:
                        for _f in glob.glob(os.path.join(_ckpt_proc, '*.' + _e)):
                            _ckpt_search = _f
                            EBOOK = _f
                            print(f"Found ebook in Drive checkpoint: {_f}")
if not EBOOK:
    raise SystemExit("No ebook found. Upload it in cell (1) or via the Files panel, then re-run.")

ebook_path = os.path.abspath(EBOOK)
os.makedirs('/content/out', exist_ok=True)

# --- Drive checkpoints: let an interrupted conversion resume after a runtime
# reset. The session id is derived from the book name, so re-running with the
# same book continues from the last synced sentence instead of starting over.
import hashlib, shutil, threading
SESSION_ID = hashlib.md5(os.path.basename(ebook_path).encode()).hexdigest()
_proc_dir = os.path.abspath('ebook2audiobook/tmp/proc-' + SESSION_ID)
_trans_dir = os.path.abspath('ebook2audiobook/tmp/translation')
CKPT = ''
if os.path.isdir('/content/drive/MyDrive'):
    CKPT = '/content/drive/MyDrive/ebook2audiobook/checkpoints/' + SESSION_ID
    for _src, _dst in ((CKPT + '/proc', _proc_dir), (CKPT + '/translation', _trans_dir)):
        if os.path.isdir(_src) and not os.path.isdir(_dst):
            shutil.copytree(_src, _dst)
            print('Checkpoint restored from Drive:', _dst)
    # Restore ebook file from checkpoint if not present in /content
    for _ebook_f in glob.glob(CKPT + '/*.epub') + glob.glob(CKPT + '/*.pdf') + \
                       glob.glob(CKPT + '/*.mobi') + glob.glob(CKPT + '/*.txt'):
        _dst_ebook = os.path.join('/content', os.path.basename(_ebook_f))
        if not os.path.exists(_dst_ebook):
            shutil.copy2(_ebook_f, _dst_ebook)
            print('Ebook restored from Drive checkpoint:', _dst_ebook)

def _sync_ckpt():
    if not CKPT:
        return
    for _src, _dst in ((_proc_dir, CKPT + '/proc'), (_trans_dir, CKPT + '/translation')):
        if os.path.isdir(_src):
            os.makedirs(_dst, exist_ok=True)
            subprocess.run(['rsync', '-a', _src + '/', _dst + '/'],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    # Also sync the ebook file so it survives runtime resets
    _ebook_src = globals().get('ebook_path', '')
    if _ebook_src and os.path.isfile(_ebook_src):
        _ebook_dst = os.path.join(CKPT, os.path.basename(_ebook_src))
        if not os.path.isfile(_ebook_dst) or os.path.getmtime(_ebook_src) > os.path.getmtime(_ebook_dst):
            shutil.copy2(_ebook_src, _ebook_dst)

_sync_stop = threading.Event()

def _sync_loop():
    while not _sync_stop.wait(180):
        _sync_ckpt()

if CKPT:
    threading.Thread(target=_sync_loop, daemon=True).start()
    print('Drive checkpoints ON: progress is synced to Drive every 3 minutes.')

cmd = [
    "python", "app.py", "--headless", "--script_mode", "full_docker",
    "--ebook", ebook_path,
    "--language", str(CFG.get('language', 'eng')),
    "--output_format", str(CFG.get('output_format', 'm4b')),
    "--device", "gpu",
    "--output_dir", "/content/out",
    "--session", SESSION_ID,
]

engine = CFG.get('tts_engine')
if engine:
    cmd += ["--tts_engine", str(engine)]

# Resolve the selected voice by name inside the cloned repo
voice_name = CFG.get('voice')
if voice_name:
    matches = [m for m in glob.glob(f"ebook2audiobook/voices/**/{voice_name}.*", recursive=True)
               if '__sessions' not in m]
    # Prefer .wav: some voices also ship engine-specific files (e.g. bark .npz)
    matches.sort(key=lambda m: (not m.endswith('.wav'), len(m)))
    if matches:
        cmd += ["--voice", os.path.abspath(matches[0])]
        print("Using voice:", matches[0])
    else:
        print(f"Voice '{voice_name}' not found in repo; using the engine default.")

for key, flag in [('speed', '--speed'), ('temperature', '--temperature'), ('repetition_penalty', '--repetition_penalty')]:
    val = CFG.get(key)
    if val is not None:
        cmd += [flag, str(val)]

if CFG.get('translate'):
    cmd += [
        "--translate",
        "--source_lang", str(CFG.get('source_lang', 'eng')),
        "--target_lang", str(CFG.get('target_lang', CFG.get('language', 'eng'))),
        "--translation_method", str(CFG.get('translation_method', 'google')),
    ]

print("\nRunning:", ' '.join(cmd))
# Stream the conversion log right into the cell (plain subprocess.run output
# is often swallowed by Colab) and keep a copy in /content/conversion.log.
_log = open('/content/conversion.log', 'w', encoding='utf-8')
_p = subprocess.Popen(cmd, cwd='ebook2audiobook', stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, bufsize=1,
                      errors='replace')
for _line in _p.stdout:
    print(_line, end='')
    _log.write(_line)
_p.wait()
_log.close()
_sync_stop.set()
print("\napp.py exit code:", _p.returncode)

# --- Ensure the translation sidecar is in /content/out ---
# The cloud clones the repo from GitHub, so the app.py that runs there may
# be an older version that does NOT copy the translation into --output_dir.
# To be safe regardless of which app.py runs, we copy any translated file
# produced inside the repo tmp/ folder into /content/out ourselves.
trans_src_dirs = [os.path.abspath("ebook2audiobook/tmp/translation"),
                  os.path.abspath("ebook2audiobook/tmp")]
for _tsd in trans_src_dirs:
    if not os.path.isdir(_tsd):
        continue
    for _tsf in glob.glob(_tsd + "/translated_*"):
        if os.path.isfile(_tsf):
            try:
                os.makedirs("/content/out", exist_ok=True)
                shutil.copy2(_tsf, "/content/out")
                print("Copied translation to /content/out:", os.path.basename(_tsf))
            except Exception as _e:
                print("Could not copy translation to /content/out:", _e)

# Collect and offer the produced audio for download
outputs = []
for ext in ('m4b', 'mp3', 'wav', 'm4a', 'flac', 'ogg', 'aac', 'opus'):
    outputs += glob.glob(f"/content/out/**/*.{ext}", recursive=True)
    outputs += glob.glob(f"ebook2audiobook/audiobooks/**/*.{ext}", recursive=True)
outputs = sorted(set(outputs), key=os.path.getmtime)
if outputs:
    result = outputs[-1]
    print("\nAudiobook ready:", result)
    if CKPT:
        shutil.rmtree(CKPT, ignore_errors=True)
        print("Drive checkpoint cleaned up.")
    # Keep a copy on Google Drive if it is mounted - it survives disconnects,
    # so nothing is lost even if the runtime dies right after the conversion.
    try:
        import shutil
        if os.path.isdir('/content/drive/MyDrive'):
            _ddir = '/content/drive/MyDrive/ebook2audiobook'
            os.makedirs(_ddir, exist_ok=True)
            shutil.copy2(result, _ddir)
            print("Saved to Google Drive:", _ddir + '/' + os.path.basename(result))
    except Exception as e:
        print("Could not copy to Drive:", e)
    # --- Translation sidecar -------------------------------------------
    # app.py copies the translated ebook (.md, or .epub for non-PDF) into
    # /content/out (the --output_dir). Offer it for download and keep a copy
    # on Google Drive too, right next to the audiobook.
    sidecars = []
    for ext in ('md', 'epub', 'txt'):
        sidecars += glob.glob(f"/content/out/**/translated_*.{ext}", recursive=True)
    sidecars = sorted(set(sidecars))
    for sc in sidecars:
        print("\nTranslation ready:", sc)
        try:
            if os.path.isdir('/content/drive/MyDrive'):
                shutil.copy2(sc, '/content/drive/MyDrive/ebook2audiobook/')
                print("Saved to Google Drive:", '/content/drive/MyDrive/ebook2audiobook/' + os.path.basename(sc))
        except Exception as e:
            print("Could not copy translation to Drive:", e)
    try:
        from google.colab import files as _files
        _files.download(result)
        for sc in sidecars:
            _files.download(sc)
    except Exception as e:
        print("Auto-download failed:", e, "- find your file at", result)
else:
    if CKPT:
        _sync_ckpt()
        print("\nProgress saved to Drive checkpoint:", CKPT)
        print("After a disconnect: delete the runtime, reopen the notebook and")
        print("re-run cells (1)-(3) - the conversion resumes where it stopped.")
    print("\nNo audio output found. Last lines of /content/conversion.log:")
    with open('/content/conversion.log', encoding='utf-8', errors='replace') as _f:
        for _line in _f.readlines()[-60:]:
            print(_line, end='')